# Lab 7: Customer Feedback Analysis

**Business question:** Can a single sentiment score explain which product feature needs action?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | The 24 customer messages from Lab 1 | about 35 min | Run outputs, saved files and the checkpoint |
| 2 Practice | Clothing reviews and a reviewed sample of 120 reviews | about 45–60 min | Three exercises and the interpretation |
| 3 Challenge | Both | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

You compare three ways to label customer text: a **sentiment lexicon** (a word list with scores), an **embedding classifier**, and **cached LLM labels**. You do not call an LLM. I ran the prompt once and saved every answer in `data/llm_outputs`, so everyone gets the same labels.

## Part 1 Guided core

### 1.1 Load the libraries

Select the `ban6403` kernel, then run the cell below. It loads the libraries we use in this lab.

In [ ]:
from pathlib import Path   # file and folder paths
import json                # reads the saved LLM records
import re                  # regular expressions: splits text into words
import numpy as np         # arrays of numbers: the vectors
import pandas as pd        # tables
from sklearn.linear_model import LogisticRegression  # the classifier in 2.7

### 1.2 Set the data and output folders

The notebook runs from the `notebooks` folder, so the data is one folder up, in `../data`. Your own results go to `../outputs/m07`.

In [ ]:
# Folder paths, relative to the notebooks folder.
DATA = Path("../data")
OUTPUT_DIR = Path("../outputs/m07")
# Create the output folder if it does not exist yet.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 Load the customer messages and the follow-up decision

We use the 23 customer messages from Lab 1 that have text (FB022 is empty). Each one has a star rating.

The service team's decision is **follow-up**: does the message report an open problem or ask for help? I read each message and listed the ones that need follow-up. This is the decision we care about. Stars and sentiment are only possible clues to it.

How this connects to Module 2: there you labeled messages `review`, `no_review` or `unclear`. Here the labels become one yes/no decision. `review` and `unclear` become follow-up = yes (an unclear message still needs a reply that asks for details), and `no_review` becomes no. My list below covers all 23 messages, including the protected ones, and replaces your own Module 2 labels, so everyone uses the same target.

In [ ]:
# Keep the 23 messages with text, numbered 0 to 22.
feedback = pd.read_csv(DATA / "raw/customer_feedback_m01.csv")
feedback = feedback.dropna(subset=["text"]).reset_index(drop=True)

# The messages that need follow-up; isin gives True for these IDs and False for the rest.
NEEDS_FOLLOW_UP = ["FB001", "FB003", "FB006", "FB008", "FB009", "FB012",
                   "FB014", "FB015", "FB017", "FB018", "FB023"]
feedback["follow_up"] = feedback["feedback_id"].isin(NEEDS_FOLLOW_UP)
print("messages:", len(feedback), "| need follow-up:", feedback["follow_up"].sum())
feedback[["feedback_id", "stars", "follow_up", "text"]].head(8)

### 1.4 Load the sentiment lexicon

A **sentiment lexicon** is a list of words, each with a score called a **valence**. Here the scores come from the VADER lexicon: people rated each word from about -4 (very negative) to +4 (very positive), and the file keeps the average. A few tokens appear twice in the file; the dictionary below keeps the last value.

Look at the scores for these words. Which ones surprise you?

In [ ]:
# The file is tab-separated (sep="\t").
lexicon = pd.read_csv(DATA / "raw/lexicons/vader_lexicon_course.tsv", sep="\t")

# A dictionary from word to score, for example valence["love"].
valence = dict(zip(lexicon["token"], lexicon["valence"]))
print("words in the lexicon:", len(valence))

# .get(word) returns None for a word that is not in the dictionary.
# {word:>8} right-aligns the word in 8 characters, so the scores line up.
for word in ["love", "great", "missing", "failed", "help", "please", "no", "huge"]:
    print(f"{word:>8}: {valence.get(word)}")

### 1.5 Score a message with the lexicon

The scoring rule is simple:

1. Lower-case the text and split it into words.
2. Look up each word in the lexicon. Words that are not in the lexicon count as 0.
3. **Negation rule:** if *not*, *no*, *never*, *cannot* or a word ending in *n't* appears in the three words before a lexicon word, flip that word's sign.
4. Add the scores. Above 0 reads as positive, below 0 as negative.

The helper returns the total and the matched words, so you can see where the score came from. Set `negation=False` to switch the rule off.

The rule is deliberately simple. It misses forms written without an apostrophe, such as *didnt*, *dont*, *wont* and *cant*, which are common in real complaints.

In [ ]:
NEGATORS = {"not", "no", "never", "cannot"}

def lexicon_score(text, negation=True):
    """Take a text; return its lexicon score and the matched words with their values."""
    # Lower-case the text and keep each run of letters and apostrophes as one word.
    words = re.findall(r"[a-z']+", text.lower())
    total = 0.0
    matched = []
    # enumerate gives each word together with its position i.
    for i, word in enumerate(words):
        # Words that are not in the lexicon count as 0, so skip them.
        if word not in valence:
            continue
        value = valence[word]
        if negation:
            # The (up to) three words before this one.
            before = words[max(0, i - 3):i]
            negated = False
            for previous in before:
                if previous in NEGATORS or previous.endswith("n't"):
                    negated = True
            if negated:
                value = -value
        total += value
        # :+.1f shows the value with its sign and one decimal, for example "help +1.7".
        matched.append(f"{word} {value:+.1f}")
    return round(total, 1), ", ".join(matched)

print(feedback.loc[0, "text"])
lexicon_score(feedback.loc[0, "text"])

### 1.6 Score every message, with and without negation

The negation rule changes two scores. Read both messages. In FB002, the rule reads "no further help needed" correctly, but now this resolved message scores negative. In FB015, it flips a word for the wrong reason.

In [ ]:
# Score each message twice: with the negation rule and without it.
scores = []
matched_words = []
scores_no_negation = []
for text in feedback["text"]:
    score, matched = lexicon_score(text)
    scores.append(score)
    matched_words.append(matched)
    score_no_negation, _ = lexicon_score(text, negation=False)  # _ means "not needed"
    scores_no_negation.append(score_no_negation)
feedback["lexicon"] = scores
feedback["matched"] = matched_words
feedback["lexicon_no_negation"] = scores_no_negation

# The messages whose score changed because of the negation rule.
changed = feedback[feedback["lexicon"].ne(feedback["lexicon_no_negation"])]
changed[["feedback_id", "text", "lexicon_no_negation", "lexicon", "matched"]]

### 1.7 Lexicon score, stars and the follow-up decision

Now put the three side by side. If one sentiment score could drive the decision, the messages that need follow-up would have low scores and low stars. Check whether they do.

In [ ]:
# np.sign gives 1, 0 or -1; .map turns each into a word.
feedback["lexicon_sign"] = np.sign(feedback["lexicon"]).map({1: "positive", 0: "zero", -1: "negative"})

# Count messages by follow-up decision and sign of the score.
print(pd.crosstab(feedback["follow_up"], feedback["lexicon_sign"]))
print()

# Number of messages and mean stars for each follow-up group.
print(feedback.groupby("follow_up")["stars"].agg(["count", "mean"]).round(2))

Read a few rows. FB018 has 5 stars and a high lexicon score, yet the customer is still waiting for a missing cable. FB006 asks for help, and the word *help* scores as positive.

In [ ]:
# .loc[rows, columns]: the four messages named here, with the columns we want to compare.
feedback.loc[feedback["feedback_id"].isin(["FB002", "FB006", "FB016", "FB018"]),
             ["feedback_id", "stars", "lexicon", "matched", "follow_up", "text"]]

### 1.8 Load the cached aspect labels

An **aspect** is the part of the product or service that an opinion is about: the charger, the delivery, the bill. I asked an LLM (gemini-3.5-flash-lite) to list the aspects in each message. For each aspect, the LLM gave a sentiment and an **evidence span**. An evidence span is the words copied from the message that support the label. The prompt is in `data/llm_outputs/prompts/m07_feedback_prompt.md`. The prompt gives definitions but no labeled examples, which is called **zero-shot** labeling.

**What is in a cached LLM record?** An **LLM** (large language model) reads a prompt and writes text. Here the prompt holds the instructions, the aspect definitions and one message. I sent each prompt once and saved the reply. The file is in **JSON Lines** format: each line is one JSON record. JSON is a text format for fields and lists, and `json.loads` reads one record into a Python dictionary. Each record has these fields:

- `model`, `prompt_version` and `created`: which model and prompt made the label, and when.
- `temperature`: a setting for randomness in the reply. At 0 the model gives its most likely answer, but a rerun can still come back slightly different.
- `latency_ms`: how long the call took, in milliseconds. `input_tokens` and `output_tokens` measure the size of the prompt and the reply.
- `raw_text`: exactly what the model wrote.
- `parsed`: the same reply read into fields. `error` is `None` (`null` in the file) when the reply could be read.

Because the replies are saved (a **cache**), every student compares methods on the same labels, and a rerun cannot change the comparison. The cell prints one record.

In [ ]:
# Read the file line by line; json.loads turns each line into a Python dictionary.
fb_records = []
for line in (DATA / "llm_outputs/m07_feedback_aspects_gemini.jsonl").read_text().splitlines():
    fb_records.append(json.loads(line))

# Print the main fields of the first record.
record = fb_records[0]
for key in ["id", "model", "prompt_version", "temperature", "created", "latency_ms", "input_tokens", "output_tokens", "error"]:
    print(f"{key}: {record[key]}")
print("raw_text (what the model wrote):")
print(record["raw_text"])
print("parsed (the reply as Python fields):", record["parsed"])

Now turn all records into one row per aspect mention.

In [ ]:
print("records:", len(fb_records), "| model:", fb_records[0]["model"], "| prompt:", fb_records[0]["prompt_version"])
# One row per aspect mention: the message ID plus the mention's fields (aspect, sentiment, span).
mention_rows = []
for fb_record in fb_records:
    for mention in fb_record["parsed"]["mentions"]:
        row = {"feedback_id": fb_record["id"]}
        row.update(mention)  # add the fields of the mention to the row
        mention_rows.append(row)
mentions = pd.DataFrame(mention_rows)

# Add the message text to each mention.
mentions = mentions.merge(feedback[["feedback_id", "text"]], on="feedback_id")
print("aspect mentions:", len(mentions))

### 1.9 Check every evidence span

An evidence span is only evidence if it really appears in the message. We check this with an exact substring test: the span must appear character for character in the text.

In [ ]:
# `span in text` is True when the span appears exactly, character for character, in the text.
span_found = []
for span, text in zip(mentions["evidence_span"], mentions["text"]):
    span_found.append(span in text)
mentions["span_found"] = span_found

# Summing True/False values counts the True ones.
print("spans found in the text:", mentions["span_found"].sum(), "of", len(mentions))

### 1.10 Mixed opinions: one message, several aspects

A **mixed opinion** contains praise and criticism, across different aspects or within one aspect. A single score for the whole message hides this. Look at the four mixed messages below. In the clothing labels of Part 2, `mixed` means both sentiments toward the same aspect.

In [ ]:
# The four messages with mixed opinions, one row per aspect mention.
mixed = ["FB001", "FB013", "FB018", "FB019"]
mentions.loc[mentions["feedback_id"].isin(mixed), ["feedback_id", "aspect", "sentiment", "evidence_span"]]

### 1.11 An aspect evidence table with denominators

Now count by aspect. The **denominator** is the number you divide by, and a report should always show it. The sentiment columns count mentions. The `messages` column counts distinct messages, out of the 23 messages with text. FB013 and FB019 mention device performance twice, once positive and once negative, so the mentions add up to more than the messages.

In [ ]:
# Mentions by aspect and sentiment.
aspect_table = pd.crosstab(mentions["aspect"], mentions["sentiment"])
# nunique counts distinct messages per aspect; out_of is the denominator.
aspect_table["messages"] = mentions.groupby("aspect")["feedback_id"].nunique()
aspect_table["out_of"] = len(feedback)
aspect_table

### 1.12 Read the LLM labels critically

The LLM labels are a draft, not the truth. Compare these two rows with the message text. FB016 says the old unit **failed**. Is that in the labels? FB024 says the case can be closed. Is that about accessories?

In [ ]:
# The aspect mentions of FB016 and FB024, next to the message text.
mentions.loc[mentions["feedback_id"].isin(["FB016", "FB024"]), ["feedback_id", "text", "aspect", "sentiment", "evidence_span"]]

### 1.13 Save your outputs

This writes two small CSV files to `outputs/m07`. The data files are not changed. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
save_cols = ["feedback_id", "stars", "follow_up", "lexicon", "lexicon_no_negation", "matched", "text"]
feedback[save_cols].to_csv(OUTPUT_DIR / "feedback_lexicon.csv", index=False)
mentions.to_csv(OUTPUT_DIR / "feedback_aspects.csv", index=False)

print("saved:", OUTPUT_DIR / "feedback_lexicon.csv")
print("saved:", OUTPUT_DIR / "feedback_aspects.csv")

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. FB018 has 5 stars and a positive lexicon score. Does it need follow-up? What does this show about using one score for the decision?
2. Name one message where the negation rule helped and one where it flipped a word for the wrong reason.

**Your answer:**

## Part 2 Practice: clothing reviews

Now you work with real customer reviews: the Women's E-Commerce Clothing Reviews in `data/course/clothing_reviews.csv`. Each of the 22,634 reviews has a star rating (1 to 5) and a department, such as Dresses or Tops. It also has a yes/no answer to "would you recommend this item?"

**Business question:** the retailer wants to know which product aspects go with low ratings and with "not recommended", in each department. A report must show its denominators.

The **reviewed sample** has 120 reviews from the `valid` split. It covers five departments (Tops, Dresses, Bottoms, Intimate, Jackets) and three rating bands (1–2, 3 and 4–5 stars), with 8 reviews in each group. So low ratings are much more common in the sample than in the data. Its labels were made by reading each review against a short codebook. The cached LLM labels were one of three readings, and where the readings differed, the majority decided. Because these LLM outputs helped determine the reference labels, this comparison is not an independent evaluation of the LLM. We cannot tell how much this raises its agreement. For each review the labels give:
- an `overall_sentiment`: `positive`, `negative` or `mixed`;
- for each of six aspects (`fit_size`, `fabric_quality`, `style_look`, `comfort`, `color`, `price_value`): the sentiment toward it (`positive`, `negative`, `mixed`, or empty when the review gives no opinion on it);
- for each labeled aspect, an evidence span in the column `<aspect>_span`.

One review (116) gives no opinion on any aspect, so all its aspect cells are empty.

The codebook with definitions, tie-break rules and edge cases is `data/llm_outputs/prompts/m07_review_codebook.md`. Read it before you start. The reviewed labels are the reference for every comparison below. They are a careful reference, not ground truth, so they can also be wrong.

### 2.1 Load the reviews and the reviewed sample

`band` groups the star ratings into the three bands of the sample design. In the reviewed file, an empty cell means "no opinion on this aspect". `keep_default_na=False` keeps those cells as empty text (`""`), so we can test for them with `.ne("")`.

In [ ]:
reviews = pd.read_csv(DATA / "course/clothing_reviews.csv")
# pd.cut puts each rating in a band: (0, 2] = 1-2 stars, (2, 3] = 3 stars, (3, 5] = 4-5 stars.
reviews["band"] = pd.cut(reviews["rating"], [0, 2, 3, 5], labels=["1-2 stars", "3 stars", "4-5 stars"])

# The reviewed labels, with empty cells kept as "" (no opinion).
reviewed = pd.read_csv(DATA / "reviewed/m07_reviewed_sample.csv", keep_default_na=False)
# Add the department, rating, band and text of each reviewed review.
review_info = reviews[["review_id", "department", "rating", "band", "recommended", "split", "review_text"]]
reviewed = reviewed.merge(review_info, on="review_id")
print("reviews:", len(reviews), "| reviewed rows:", len(reviewed), "| splits:", reviewed["split"].unique().tolist())
# The sample design: reviews per department and band.
pd.crosstab(reviewed["department"], reviewed["band"])

### 2.2 The reviewed aspects and their evidence

The file has one row per review and two columns per aspect. For counting it is easier to have one row per **aspect mention**. `melt` turns the six aspect columns into rows. Then we keep the rows with a sentiment and check every span with the exact substring test from 1.9. The table at the end counts the mentions of each aspect by sentiment.

In [ ]:
ASPECTS = ["fit_size", "fabric_quality", "style_look", "comfort", "color", "price_value"]
# melt: one row per review and aspect. The aspect name goes to "aspect", its label to "sentiment".
reviewed_mentions = reviewed.melt(
    id_vars=["review_id", "department", "band", "review_text"],  # columns to keep on every row
    value_vars=ASPECTS,                                          # columns to turn into rows
    var_name="aspect",
    value_name="sentiment",
)
# Keep the rows with an opinion (not empty).
reviewed_mentions = reviewed_mentions[reviewed_mentions["sentiment"].ne("")].copy()

# Look up each mention's evidence span in the column "<aspect>_span" of its review.
reviewed_by_id = reviewed.set_index("review_id")
spans = []
for review_id, aspect in zip(reviewed_mentions["review_id"], reviewed_mentions["aspect"]):
    spans.append(reviewed_by_id.loc[review_id, aspect + "_span"])
reviewed_mentions["evidence_span"] = spans

# The exact substring test from 1.9.
found = []
for span, text in zip(reviewed_mentions["evidence_span"], reviewed_mentions["review_text"]):
    found.append(span in text)
print("aspect mentions:", len(reviewed_mentions), "| spans found:", sum(found), "of", len(found))

# How many reviews have each overall sentiment, then mentions by aspect and sentiment.
print(reviewed["overall_sentiment"].value_counts().to_string())
pd.crosstab(reviewed_mentions["aspect"], reviewed_mentions["sentiment"])

### 2.3 Lexicon scores for all reviews

Score every review with the same lexicon and negation rule as in Part 1. If the score measured how happy a customer is, it would rise with the star rating, and low-rated reviews would score below 0. The table shows, for each rating, how many reviews there are, the median score and how many score above 0.

In [ ]:
# Score every review with the Part 1 helper (about 22,000 reviews, a few seconds).
lexicon_scores = []
for text in reviews["review_text"]:
    score, _ = lexicon_score(text)
    lexicon_scores.append(score)
reviews["lexicon"] = lexicon_scores

# For each rating: number of reviews, median score and number of reviews scoring above 0.
grouped = reviews.groupby("rating")["lexicon"]
above_zero = reviews["lexicon"] > 0
by_rating = pd.DataFrame({
    "n": grouped.size(),
    "median": grouped.median(),
    "above_zero": above_zero.groupby(reviews["rating"]).sum(),
})
by_rating["share_above"] = (by_rating["above_zero"] / by_rating["n"]).round(2)

# True for the 1-2 star reviews.
low = reviews["rating"] <= 2
print("1-2 star reviews scored above 0:", (reviews.loc[low, "lexicon"] > 0).sum(), "of", low.sum())
by_rating

### 2.4 Where the lexicon disagrees with the stars

The median score rises with the rating, so the lexicon carries some signal. But most low-rated reviews still score above 0. Which words push them up? The cell counts low-rated (1–2 star) and high-rated (4–5 star) reviews that contain each word or phrase. *Huge* is in the lexicon as a positive word, but in clothing reviews it usually means the size is wrong. The phrase *wanted to love* has no score of its own, yet *love* scores +3.2.

In [ ]:
lowered = reviews["review_text"].str.lower()
high = reviews["rating"] >= 4

rows = []
for term in ["love", "cute", "huge", "disappointed", "wanted to love"]:
    # \b marks a word boundary, so "love" does not match "lovely".
    has = lowered.str.contains(rf"\b{term}\b")
    rows.append({
        "term": term,
        "valence": valence.get(term),
        "low_rated": has[low].sum(),
        "of_low": low.sum(),
        "high_rated": has[high].sum(),
        "of_high": high.sum(),
    })
print(pd.DataFrame(rows).to_string(index=False))

# Reviews that contain the phrase "wanted to love", and how many of them score above 0.
wanted = lowered.str.contains("wanted to love")
print("'wanted to love' reviews:", wanted.sum(), "| scored above 0:", (reviews.loc[wanted, "lexicon"] > 0).sum())

Read the two low-rated reviews with the highest scores. The matched words show where the score came from.

In [ ]:
# The two 1-2 star reviews with the highest lexicon scores.
top_low = reviews[low].sort_values("lexicon", ascending=False).head(2)
for review_id, rating, text in zip(top_low["review_id"], top_low["rating"], top_low["review_text"]):
    score, matched = lexicon_score(text)
    print(f"review {review_id} | {rating} stars | score {score}")
    print(text)
    print("matched:", matched)
    print()

### 2.5 Does the lexicon score match the reviewed sentiment?

Now compare with the reviewed labels instead of the stars. If the lexicon read the writer's verdict, `negative` reviews would score below 0.

In [ ]:
# Add the lexicon score to each reviewed review.
reviewed = reviewed.merge(reviews[["review_id", "lexicon"]], on="review_id")

# For each reviewed sentiment: number of reviews, median score and number scoring above 0.
grouped = reviewed.groupby("overall_sentiment")["lexicon"]
above_zero = reviewed["lexicon"] > 0
pd.DataFrame({
    "count": grouped.count(),
    "median": grouped.median(),
    "above_zero": above_zero.groupby(reviewed["overall_sentiment"]).sum(),
})

### 2.6 Weak labels from the star rating

A lexicon gives no aspect, and its zero line does not separate low and high ratings. Can a classifier learn the verdict from the text instead? It needs labeled examples, and we do not have 13,581 hand-labeled reviews. Instead we use **weak labels**: labels made by a cheap rule that is often, but not always, right.

Our rule uses the stars: 1–2 stars → `negative`, 4–5 stars → `positive`. Three-star reviews are left out, because their verdict is unclear. We label only the `train` split.

In [ ]:
# The weak-label rule. .map leaves 3-star reviews empty, because 3 is not in the dictionary.
STARS_TO_LABEL = {1: "negative", 2: "negative", 4: "positive", 5: "positive"}
reviews["weak_label"] = reviews["rating"].map(STARS_TO_LABEL)

# Training reviews that have a weak label.
is_train = reviews["split"].eq("train") & reviews["weak_label"].notna()
print("train reviews with a weak label:", is_train.sum(), "of", reviews["split"].eq("train").sum())
reviews.loc[is_train, "weak_label"].value_counts()

How good are the weak labels? Compare them with the reviewed `overall_sentiment` for the reviewed reviews that are not 3-star.

In [ ]:
# The same rule on the reviewed sample; keep the reviews that are not 3-star.
reviewed["weak_label"] = reviewed["rating"].map(STARS_TO_LABEL)
has_weak = reviewed["weak_label"].notna()
with_weak = reviewed[has_weak]
print("reviewed reviews with a weak label:", has_weak.sum(), "of", len(reviewed))

# How often the weak label equals the reviewed sentiment, then the full comparison.
print("weak label equals reviewed sentiment:", (with_weak["weak_label"] == with_weak["overall_sentiment"]).sum())
pd.crosstab(with_weak["overall_sentiment"], with_weak["weak_label"])

### 2.7 Train an embedding classifier on the weak labels

We use the saved sentence embeddings for the clothing reviews as features. They are the same vectors you used in Lab 5 (`data/embeddings/clothing_minilm.npy`), one per review in file order. We fit a logistic regression on the weakly labeled `train` reviews. Then we predict a label for the `valid` reviews, including the reviewed sample. The model never saw them.

**Fit on train, predict on valid.** A classifier learns from labeled examples. **Logistic regression** is a simple, widely used classifier. `fit` learns one weight per embedding dimension, so that the weak labels are predicted as well as possible. `predict` then gives the more likely label for reviews the model has not seen. This is the same discipline as fit and transform in Module 4: learn only from `train`, then apply the result to `valid`. Module 8 explains the classifier in more detail.

The weak labels are mostly `positive`. So we also print the **majority baseline**: how often you would be right by always answering `positive`.

**Selecting rows of an array.** `vectors` is a two-dimensional array: one row per review and one column per embedding feature. `to_numpy()` turns a pandas True/False column into an array, and `vectors[mask]` keeps the rows marked True. `vectors[reviewed_rows]` does the same with a list of row positions: it keeps those rows, in that order.

In [ ]:
# One saved vector per review, in the row order of clothing_reviews.csv (as in Lab 5).
# The file stores compact 16-bit numbers; .astype("float32") turns them into ordinary numbers for the classifier.
vectors = np.load(DATA / "embeddings/clothing_minilm.npy").astype("float32")

# max_iter=1000 allows enough steps for the fit to finish.
clf = LogisticRegression(max_iter=1000)
# fit learns from the training rows only: their vectors (X) and their weak labels (y).
# vectors[is_train.to_numpy()] keeps the rows of the training reviews with a weak label.
clf.fit(vectors[is_train.to_numpy()], reviews.loc[is_train, "weak_label"])

# predict gives one label per validation review the model has not seen.
is_valid = reviews["split"].eq("valid") & reviews["weak_label"].notna()
valid_pred = clf.predict(vectors[is_valid.to_numpy()])
valid_weak = reviews.loc[is_valid, "weak_label"]
print("valid reviews, prediction equals weak label:", (valid_pred == valid_weak).sum(), "of", is_valid.sum())
print("majority baseline (always positive):", valid_weak.eq("positive").sum(), "of", is_valid.sum())

# row_of[review_id] gives a review's row position, which is also its row in `vectors`.
row_of = pd.Series(range(len(reviews)), index=reviews["review_id"])
# The row positions of the 120 reviewed reviews; vectors[reviewed_rows] keeps those rows in that order.
reviewed_rows = row_of[reviewed["review_id"]].to_numpy()
reviewed["clf_label"] = clf.predict(vectors[reviewed_rows])
print("reviewed sample, classifier equals reviewed sentiment:", (reviewed["clf_label"] == reviewed["overall_sentiment"]).sum(), "of", len(reviewed))

### 2.8 Load the cached LLM aspect labels

I sent 360 reviews to gemini-3.5-flash-lite with the prompt `m07_review_aspects_v1` (in `data/llm_outputs/prompts/`). The 360 are the 120 reviewed reviews plus 240 more from `train` and `valid`: 24 per department and rating band. The prompt sends only the review text, with the codebook's aspect definitions and rules but no labeled examples. Each record has the fields from 1.8: model, prompt version, date, temperature, time taken, token counts, raw text, parsed fields and error. It also has two fields that we do not use here.

The prompt asks for each aspect at most once per review. The LLM did not always follow that rule. The cell counts the repeats. Then it builds one row per review and aspect: when the LLM gave the same aspect twice with different sentiments, the row says `mixed`. A review with no aspect mention gets empty cells.

In [ ]:
# Read the 360 saved records, one JSON record per line, as in 1.8.
llm_records = []
for line in (DATA / "llm_outputs/m07_review_aspects_gemini.jsonl").read_text().splitlines():
    llm_records.append(json.loads(line))

# Count the records whose reply could not be read, and collect the time each call took.
n_errors = 0
latencies = []
for rec in llm_records:
    if rec["error"] is not None:
        n_errors += 1
    latencies.append(rec["latency_ms"])
first = llm_records[0]
print("records:", len(llm_records), "| errors:", n_errors)
print("model:", first["model"], "| prompt:", first["prompt_version"], "| temperature:", first["temperature"], "| date:", first["created"][:10])
print("median seconds per review:", np.median(latencies) / 1000)

# One row per aspect mention, with the review's department, band and text.
mention_rows = []
for rec in llm_records:
    for mention in rec["parsed"]["mentions"]:
        row = {"review_id": int(rec["id"])}
        row.update(mention)
        mention_rows.append(row)
llm_mentions = pd.DataFrame(mention_rows)
llm_mentions = llm_mentions.merge(reviews[["review_id", "department", "band", "review_text"]], on="review_id")

# duplicated is True for a review-aspect pair that already appeared above.
n_repeats = llm_mentions.duplicated(["review_id", "aspect"]).sum()
print("LLM aspect mentions:", len(llm_mentions), "| repeated review-aspect pairs:", n_repeats)

def combine(sentiments):
    """Take the sentiments one review gave one aspect; return the sentiment, or "mixed" if they differ."""
    if sentiments.nunique() == 1:
        return sentiments.iloc[0]
    return "mixed"

# The review IDs in file order.
record_ids = []
for rec in llm_records:
    record_ids.append(int(rec["id"]))

# One sentiment per review and aspect: agg passes each group's sentiment column to combine.
by_review_aspect = llm_mentions.groupby(["review_id", "aspect"])["sentiment"].agg(combine)
# unstack turns the aspects (the second row label) into columns.
llm_aspects = by_review_aspect.unstack()
# reindex gives one row per record and one column per aspect; missing cells become "".
llm_aspects = llm_aspects.reindex(index=record_ids, columns=ASPECTS).fillna("")
# .all(axis=1) is True for a row where every cell is "".
print("reviews with no aspect mention:", (llm_aspects == "").all(axis=1).sum())

# The LLM's overall sentiment for each review ID.
overall_by_id = {}
for rec in llm_records:
    overall_by_id[int(rec["id"])] = rec["parsed"]["overall_sentiment"]
llm_overall = pd.Series(overall_by_id, name="llm_overall")

# Add the LLM labels to the reviewed sample; add_prefix renames fit_size to llm_fit_size, and so on.
reviewed = reviewed.merge(llm_aspects.add_prefix("llm_"), left_on="review_id", right_index=True)
reviewed["llm_overall"] = reviewed["review_id"].map(llm_overall)
reviewed[["review_id", "overall_sentiment", "llm_overall", "fit_size", "llm_fit_size", "fabric_quality", "llm_fabric_quality"]].head(5)

### Exercise 1: compare the three methods on the same reviewed sample

All three methods are judged on the same 120 reviews. They give different outputs, so compare each one where it has an output.

**a. Overall sentiment (all three methods).** Make a lexicon label: `negative` if the score is below 0, otherwise `positive`. Then count, out of 120, how often each of these equals the reviewed `overall_sentiment`:
- the lexicon label;
- `clf_label` (the classifier);
- `llm_overall` (the LLM).

Show `pd.crosstab` of the reviewed sentiment against each. How many `mixed` reviews can the lexicon and the classifier get right?

Also report the most common reviewed sentiment and its count. Does each method beat always choosing that sentiment?

**b. Aspects (LLM only).** The lexicon and the classifier give no aspect. For each aspect, count:
- `reviewed`: reviews where the reviewed label has that aspect (not empty);
- `llm`: reviews where the LLM has it;
- `both`: reviews where both have it;
- `same_sentiment`: reviews where both have it with the same sentiment.

Hint: for a, `np.where(reviewed["lexicon"] < 0, "negative", "positive")` makes the lexicon label. For b, loop over `ASPECTS`; `reviewed[aspect].ne("")` is True where the reviewed label has the aspect, and `reviewed["llm_" + aspect]` holds the LLM's sentiment.

In [ ]:
# Exercise 1: your code here
# a. TODO: make the lexicon label with np.where(reviewed["lexicon"] < 0, "negative", "positive").
#    TODO: for the lexicon label, "clf_label" and "llm_overall", count how often each equals
#          reviewed["overall_sentiment"] (out of 120) and show pd.crosstab of the two.
#    TODO: print the most common reviewed sentiment and its count (value_counts), the baseline.
# b. TODO: loop over ASPECTS and count, for each aspect: reviewed, llm, both, same_sentiment.
#    TODO: put the counts in one table, one row per aspect.

### Exercise 2: check the LLM evidence and read the disagreements

1. For every row of `llm_mentions` (all 360 reviews), test whether `evidence_span` appears **exactly** in `review_text` (as in 2.2). Print how many pass, out of how many. Print the spans that fail. What did the LLM do to them?
2. On the reviewed sample, show the reviews where `llm_fit_size` differs from the reviewed `fit_size`, with `review_id`, `fit_size`, `fit_size_span`, `llm_fit_size` and the text.
3. Read two of them in full and decide who you agree with, using the codebook.

Hint: rule 3 of the codebook ("runs small" is negative even when another size fits) explains several disagreements.

In [ ]:
# Exercise 2: your code here
# TODO: test every evidence_span of llm_mentions against its review_text (exact `in` test, as in 2.2);
#       print how many pass, out of how many, and print the spans that fail.
# TODO: show the reviewed rows where llm_fit_size differs from fit_size, with review_id, fit_size,
#       fit_size_span, llm_fit_size and the text.
# TODO: print two of these reviews in full, then write who you agree with in the interpretation.

### Exercise 3: report one aspect pattern with denominators

Do fit problems go with low ratings in every department? Use the LLM labels for all 360 reviews (`llm_mentions`). For each department, count the reviews with a `negative` or `mixed` `fit_size` mention:
- among the 1–2 star reviews (24 per department);
- among the 4–5 star reviews (24 per department).

Count each review once, even when the LLM gave the aspect twice. The denominator is all 24 sampled reviews in the group, including those without that aspect. Show the counts with their denominators. Do the same for `fabric_quality`. Then write one sentence in the form "In k of n 1–2 star ... reviews, ..." for the department with the highest fit count, and say whose labels produced it.

Remember that this sample has 24 reviews per department and band. The share of low ratings in the sample is set by the design, so do not report a sample share as a share of all reviews.

In [ ]:
# Exercise 3: your code here
# TODO: keep the llm_mentions rows for fit_size with sentiment "negative" or "mixed".
# TODO: for each department, count the 1-2 star and the 4-5 star reviews (of the 360) that have
#       such a mention, and show each count next to its denominator (24).
#       Count reviews, not rows: a review can have two mentions of one aspect.
#       counts = selected.groupby(["department", "band"])["review_id"].nunique()
# TODO: repeat for fabric_quality, then write your sentence "In k of n 1-2 star ... reviews, ...".

### Practice interpretation

Write 3–5 sentences:

- Using 2.3–2.5, can the lexicon score tell you which reviews are negative? Give a number.
- Using 2.6, 2.7 and Exercise 1, which method agreed most often with the reviewed sentiment, and where did each struggle? Why might star-based weak labels limit the classifier?
- Using Exercises 2 and 3, would you give the LLM's aspect counts to a manager as they are? State your pattern with its denominator and one limitation.

**Your answer:**

### Submit

Run the whole notebook from the top (**Restart**, then **Run All**) and save it, so the outputs are saved in the file. Then commit and push it in GitHub Desktop before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

If you used an AI tool to explain or debug code, add one line here: the tool, what you used it for, and how you checked the result.

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. Other cutoffs for the lexicon

In Exercise 1a, a score below 0 meant `negative`. Try other cutoffs on the reviewed sample. Does any cutoff make the lexicon a good detector of negative reviews?

In [ ]:
for cutoff in [0, 3, 6, 9]:
    # np.where(condition, a, b) gives a where the condition is True and b elsewhere.
    lex_label = np.where(reviewed["lexicon"] < cutoff, "negative", "positive")
    agree = (lex_label == reviewed["overall_sentiment"]).sum()
    print(f"negative if score < {cutoff}: agrees with reviewed sentiment on {agree} of {len(reviewed)}")

### B. Balance the weak labels

Most weak labels are `positive`, so the classifier leans that way. `class_weight="balanced"` gives the rare class more weight during training. Does it help on the reviewed sample?

In [ ]:
# Same classifier as 2.7, but class_weight="balanced" gives the rare label more weight.
balanced = LogisticRegression(max_iter=1000, class_weight="balanced")
balanced.fit(vectors[is_train.to_numpy()], reviews.loc[is_train, "weak_label"])
reviewed["balanced_label"] = balanced.predict(vectors[row_of[reviewed["review_id"]].to_numpy()])
print("balanced classifier equals reviewed sentiment:", (reviewed["balanced_label"] == reviewed["overall_sentiment"]).sum(), "of", len(reviewed))
pd.crosstab(reviewed["overall_sentiment"], reviewed["balanced_label"])

### C. The full VADER rules

The `vaderSentiment` package adds rules for intensifiers ("very"), capital letters, exclamation marks and "but". It is not in the course environment. If you install it (`pip install vaderSentiment` in the `ban6403` environment), this cell scores the first five customer messages with the full rules.

In [ ]:
# The package is optional, so the cell prints a note instead of failing when it is missing.
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ImportError:
    print("vaderSentiment is not installed. This part is optional.")
else:
    analyzer = SentimentIntensityAnalyzer()
    for text in feedback["text"].head(5):
        # "compound" is VADER's overall score, from -1 (negative) to +1 (positive).
        print(round(analyzer.polarity_scores(text)["compound"], 2), text)

## Further reading

- [VADER: A Parsimonious Rule-based Model for Sentiment Analysis of Social Media Text (Hutto and Gilbert, 2014)](https://ojs.aaai.org/index.php/ICWSM/article/view/14550)
- [Jurafsky and Martin, Speech and Language Processing, chapter on lexicons for sentiment and affect](https://web.stanford.edu/~jurafsky/slp3/)
- [Aspect-based sentiment analysis: SemEval-2014 Task 4](https://aclanthology.org/S14-2004/)

Next, Module 8 trains classifiers for a decision target, routing complaints, and compares them on the same validation set. The test set stays sealed until Module 9.